# 04 — Clustering & Cell-Type Annotation

This notebook covers:
- Leiden / Louvain graph-based clustering
- Resolution tuning
- UMAP / t-SNE cluster visualisation
- Known marker gene overlay
- Manual cluster annotation


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import scanpy as sc
import matplotlib.pyplot as plt
import config as cfg
from utils import load_adata, save_adata

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor='white')
%matplotlib inline
print('Setup complete ✓')

In [ ]:
adata = load_adata(cfg.ADATA_REDUCED_PATH)
print(adata)

## 1. Leiden Clustering

In [ ]:
# Try different resolutions to find the right granularity
for res in [0.3, 0.5, 0.8, 1.0]:
    sc.tl.leiden(adata, resolution=res, key_added=f'leiden_{res}', random_state=cfg.CLUSTERING['random_state'])
    n = adata.obs[f'leiden_{res}'].nunique()
    print(f'Resolution {res}: {n} clusters')

In [ ]:
# Visualise all resolutions
sc.pl.umap(adata, color=['leiden_0.3','leiden_0.5','leiden_0.8','leiden_1.0'], ncols=2, legend_loc='on data')

In [ ]:
# Pick final resolution and copy to 'leiden'
CHOSEN_RESOLUTION = cfg.CLUSTERING['resolution']
sc.tl.leiden(adata, resolution=CHOSEN_RESOLUTION, key_added='leiden', random_state=cfg.CLUSTERING['random_state'])
print(f'Final: {adata.obs["leiden"].nunique()} clusters at resolution {CHOSEN_RESOLUTION}')

In [ ]:
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='UMAP — Leiden Clusters')

## 2. Known Marker Genes

In [ ]:
KNOWN_MARKERS = {
    'T Cells':      ['CD3D', 'CD3E'],
    'CD4+ T Cells': ['CD4', 'IL7R'],
    'CD8+ T Cells': ['CD8A', 'CD8B'],
    'B Cells':      ['MS4A1', 'CD79A'],
    'NK Cells':     ['GNLY', 'NKG7'],
    'Monocytes':    ['CD14', 'LYZ'],
    'DC':           ['FCER1A'],
}

all_markers = [g for genes in KNOWN_MARKERS.values() for g in genes]
present = [g for g in all_markers if g in adata.var_names]
print(f'{len(present)} / {len(all_markers)} markers found in dataset: {present}')

In [ ]:
if present:
    sc.pl.umap(adata, color=present, ncols=4, use_raw=False)

In [ ]:
markers_present = {ct: [g for g in genes if g in adata.var_names] for ct, genes in KNOWN_MARKERS.items()}
markers_present = {ct: g for ct, g in markers_present.items() if g}
if markers_present:
    sc.pl.dotplot(adata, markers_present, groupby='leiden')

## 3. Annotate Clusters

After inspecting marker genes from step 05, fill in the mapping below:

In [ ]:
# ── Edit this mapping after reviewing marker genes ──
ANNOTATIONS = {
    # '0': 'CD4+ T Cells',
    # '1': 'CD8+ T Cells',
    # '2': 'B Cells',
    # '3': 'NK Cells',
    # '4': 'Monocytes',
}

if ANNOTATIONS:
    adata.obs['cell_type'] = adata.obs['leiden'].map(ANNOTATIONS).fillna('Unknown').astype('category')
    sc.pl.umap(adata, color='cell_type', legend_loc='on data', title='Cell Types')

## 4. Save

In [ ]:
save_adata(adata, cfg.ADATA_CLUSTERED_PATH)
print('Done ✓')